In [ ]:
import logging

from Module_4.meta_data_enrich.comprehend.simplified_version import enrich_chunk

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)

logger = logging.getLogger(__name__)

In [1]:
input_file = '/Users/akellaprudhvi/mystuff/Course/GenAI-Course-Modules/Module_4_notebooks/semantic_chunks.json'

In [2]:
output_file = '/Users/akellaprudhvi/mystuff/Course/GenAI-Course-Modules/Module_4_notebooks/semantic_chunks_enriched.json'

In [7]:
import json

with open(input_file, 'r', encoding='utf-8') as f:
        semantic_chunks = json.load(f)

In [11]:
import boto3
region_name = 'us-east-1'
comprehend_client = boto3.client('comprehend', region_name=region_name)

In [17]:
import re
from typing import List, Dict

def init_patterns() -> Dict[str, re.Pattern]:
    """
    Initialize regex patterns for custom extraction

    PATTERNS EXPLAINED:
    -------------------

    1. Monetary Values: $15.4B, $2.3M, $500K
       Pattern: \$\s*(\d+(?:,\d{3})*(?:\.\d+)?)\s*([BMK])?

    2. Percentages: 25%, 15.5%
       Pattern: (\d+(?:\.\d+)?)\s*%

    3. Quarters: Q3 2024, Q4, Q1 FY24
       Pattern: Q[1-4]\s*(?:20\d{2}|FY\d{2}|\d{2})?

    4. Years: 2024, FY2024, CY2023
       Pattern: (?:FY|CY)?\s*20\d{2}

    5. Financial Metrics: EBITDA, EPS, ROE, revenue
       Pattern: \b(EBITDA|EPS|ROE|...)\b

    Returns
    -------
    Dict[str, re.Pattern]
        Dictionary of compiled regex patterns
    """
    return {
        'monetary_values': re.compile(
            r'\$\s*(\d+(?:,\d{3})*(?:\.\d+)?)\s*([BMK])?'
        ),
        'percentages': re.compile(
            r'(\d+(?:\.\d+)?)\s*%'
        ),
        'quarters': re.compile(
            r'Q[1-4]\s*(?:20\d{2}|FY\d{2}|\d{2})?'
        ),
        'years': re.compile(
            r'(?:FY|CY)?\s*20\d{2}'
        ),
        'financial_metrics': re.compile(
            r'\b(EBITDA|EPS|ROE|ROI|P/E|revenue|profit|margin)\b',
            re.IGNORECASE
        )
    }


# Initialize patterns globally
PATTERNS = init_patterns()

<>:12: SyntaxWarning: invalid escape sequence '\$'
<>:12: SyntaxWarning: invalid escape sequence '\$'
/var/folders/dq/76c6b9rx7tv5fhypn7n6fx800000gn/T/ipykernel_61017/292686280.py:12: SyntaxWarning: invalid escape sequence '\$'
  Pattern: \$\s*(\d+(?:,\d{3})*(?:\.\d+)?)\s*([BMK])?


In [22]:
from pprint import pprint
from collections import defaultdict
confidence_threshold = 0.8
enrich_chunks = []
for i, chunk in enumerate(semantic_chunks, 1):
    # print(chunk['content'])
    ########## Entity Detection
    response = comprehend_client.detect_entities(
                    Text=chunk['content'],
                    LanguageCode='en'
                )
    entities_by_type = defaultdict(list)
    for entity in response.get('Entities', []):
         if entity['Score'] >= confidence_threshold:
            entity_type = entity['Type']
            entities_by_type[entity_type].append({
                'text': entity['Text'],
                'score': entity['Score']
            })
    entities_by_type = dict(entities_by_type)
    # ########## Entity Detection
    # response = comprehend_client.detect_key_phrases(
    #             Text=chunk['content'],
    #             LanguageCode='en'
    #         )
    # phrases = []
    # pprint(response)
    # for phrase in response.get('KeyPhrases', []):
    ################ custom entities
    custom_entities = {}
    monetary_matches = PATTERNS['monetary_values'].findall(chunk['content'])
    custom_entities['monetary_values'] = [
        f"${amount}{suffix}" if suffix else f"${amount}"
        for amount, suffix in monetary_matches
    ]
    pct_matches = PATTERNS['percentages'].findall(chunk['content'])
    custom_entities['percentages'] = [f"{pct}%" for pct in pct_matches]
    custom_entities['quarters'] = PATTERNS['quarters'].findall(chunk['content'])
    custom_entities['years'] = PATTERNS['years'].findall(chunk['content'])
    # pprint(custom_entities)
    #################### PII detection and redacting it
    detail = comprehend_client.detect_pii_entities(Text=chunk['content'], LanguageCode='en')
    entities = detail.get('Entities', [])
    # pprint(entities)
    redacted = chunk['content']
    for entity in sorted(entities, key=lambda e: e['BeginOffset'], reverse=True):
        start    = entity['BeginOffset']
        end      = entity['EndOffset']
        pii_type = entity['Type']
        redacted = redacted[:start] + f'[REDACTED_{pii_type}]' + redacted[end:]

    chunk['metadata']['entities'] = entities_by_type
    chunk['metadata']['custom_entities'] = custom_entities
    chunk['content_sanitised'] = redacted
    enrich_chunks.append(chunk)

In [23]:
print(enrich_chunks)

[{'content': '## Thematics\n\n## Uncovering Alpha in AI\'s Rate of Change\n\nMore than two years since ChatGPT\'s launch, we remain in the early innings of AI\'s diffusion. This is the third iteration of the most comprehensive AI stock mapping exercise in the market. Rate of change continues to drive outperformance, and we believe 2025 will be the year of Agentic AI.\n\nAI\'s Rate of Change Continues to Surprise: Our first AI Adopter survey was published in January 2024, our second in June 2024. This is our third such analysis. We have been surprised by the continued extent of changes made by our analysts across >3,700 global stocks under coverage. 585 stocks   had their AI exposure or materiality changed ($13trn of market cap). AI model capabilities and costs continue to evolve rapidly and corporate adoption is still low. AI\'s diffusion is accelerating but decidedly remains in its early innings.\n\nAI\'s Rate of Change Has Driven Outperformance: Exhibit 1  shows the 2H24 outperforman

In [26]:
from datetime import datetime

output = {
        'metadata': {
            'enriched_at':      datetime.now().isoformat(),
            'total_chunks':     len(enrich_chunks),
            'enricher_version': '3.0.0'
        },
        'chunks':     enrich_chunks,
    }
with open(output_file, 'w', encoding='utf-8') as fh:
    json.dump(output, fh, indent=2, ensure_ascii=False)